# Jets (Modane): statistical validation of the generated signals

Velocity signals from the Modane jet (windows of 256 samples), against MGD samples from four models.
All models contain $L_6$ and scattering orders 1-2; they differ by two ingredients:

| model | higher orders (Scat3 Re+Im, Scat4 Mod2 Re+Im) | scalar potential on Morlet coefficients | $\lambda$ |
|---|---|---|---|
| `full` | yes | yes | 1e-2 |
| `noscalar` | yes | no | 1e-2 |
| `scat2_scalar` | no | yes | 1e-3 |
| `scat2` | no | no | 1e-4 |

5 seeds per model (500-504), pooled. Colours: blue = with higher orders, orange = without; dark/solid = with the
scalar potential, light/dashed = without. Data in black.

Figures: (1) example signals, (2) marginals of the wavelet coefficients per scale, (3) spectrum, skewness and
flatness of the increments, (4) signed leverage. Run on Jean Zay from `jets/` (needs `saved_results/`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()                                   # jets/
sys.path[:0] = [str(ROOT.parent / 'turbulence'), str(ROOT.parent), str(ROOT)]
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import run_comparison as rc
import jets_compare as jc
from check_moments import signed_leverage_correlation
%matplotlib inline
# after the imports on purpose: a module calling matplotlib.use('Agg') at import would otherwise hide every plot

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.titlesize': 10, 'axes.labelsize': 10,
                     'legend.fontsize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.2})

ARMS = {'full': 'jets_full', 'noscalar': 'jets_noscalar',
        'scat2_scalar': 'jets_scat2_scalar', 'scat2': 'jets_scat2'}      # {name: --label}
SEEDS = [500, 501, 502, 503, 504]
# colour = higher orders (blue yes / orange no), shade + line style = scalar potential (dark solid yes / light dashed no)
STYLE = {'data':         dict(color='black',   ls='-',  lw=2.2, zorder=3),
         'full':         dict(color='#08519c', ls='-',  lw=1.6),
         'noscalar':     dict(color='#6baed6', ls='--', lw=1.6),
         'scat2_scalar': dict(color='#a63603', ls='-',  lw=1.6),
         'scat2':        dict(color='#fd8d3c', ls='--', lw=1.6)}

def legend(fig, names):
    # one legend per figure, above the panels
    h = [Line2D([], [], **STYLE[n]) for n in names]
    fig.legend(h, names, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=len(names), frameon=False)

xs, cfg0 = {}, None                                 # {model: pooled samples (n_seeds * n, T)}
for arm, label in ARMS.items():
    seeds = []
    for s, cfg in jc.find_seed_configs(ROOT, label, SEEDS).items():
        if cfg is None:
            print(f'{arm} seed {s}: no saved run'); continue
        r = rc.load_config(cfg, ROOT, label=label, extras=False)
        seeds.append(r['xt'].numpy()); cfg0 = cfg0 or r['cfg']
    if seeds:
        xs[arm] = np.concatenate(seeds)
x1 = jc.reference_data(cfg0).numpy()
J = int(cfg0['J'])
sets = {'data': x1, **xs}
print('data', x1.shape, '| samples', {k: v.shape for k, v in xs.items()})

## 1. Example signals
Three random signals per source, same y-scale everywhere.

In [ ]:
n_ex = 3
rng = np.random.default_rng(0)
lim = 1.05 * max(np.percentile(np.abs(v), 99.9) for v in sets.values())
fig, axes = plt.subplots(len(sets), n_ex, figsize=(3.6 * n_ex, 1.3 * len(sets)), sharex=True, sharey=True)
for i, (name, v) in enumerate(sets.items()):
    for j, idx in enumerate(rng.choice(len(v), n_ex, replace=False)):
        axes[i, j].plot(v[idx], color=STYLE[name]['color'], lw=0.9)
    axes[i, 0].set_ylabel(name, rotation=0, ha='right', va='center')
axes[0, 0].set_ylim(-lim, lim)
for ax in axes[-1]:
    ax.set_xlabel('t (samples)')
fig.tight_layout()
plt.show()

## 2. Wavelet marginals
Histogram of $\mathrm{Re}\,W_j$, the Morlet coefficients at scale $j$ ($j=0$ finest; the bank of the scalar
potential), in units of the data's standard deviation $\sigma_j$ at that scale, log density.
Dotted grey: Gaussian of the same variance.

In [ ]:
M = x1.shape[-1]
psi, _, xi = rc.wavelet_bank(M, 'morlet', J)
W = {k: np.real(rc._coeffs(v, psi, 10000, np.random.default_rng(0))).astype(np.float32) for k, v in sets.items()}
sig = W['data'].std(axis=1)                         # (J,), data std per scale
ncol = 4
fig, axes = plt.subplots(int(np.ceil(J / ncol)), ncol, figsize=(3.4 * ncol, 2.8 * np.ceil(J / ncol)),
                         squeeze=False, sharey=True)
for j, ax in enumerate(axes.flat):
    if j >= J:
        ax.axis('off'); continue
    L = 1.1 * max(np.abs(w[j]).max() for w in W.values()) / sig[j]
    bins = np.linspace(-L, L, 121)
    c = 0.5 * (bins[1:] + bins[:-1])
    for name, w in W.items():
        h, _ = np.histogram(w[j] / sig[j], bins=bins, density=True)
        ax.plot(c[h > 0], h[h > 0], **STYLE[name])
    ax.plot(c, np.exp(-c ** 2 / 2) / np.sqrt(2 * np.pi), ':', color='0.5', lw=1)
    ax.set_yscale('log'); ax.set_ylim(bottom=1e-6)
    ax.set_title(f'j = {j}  (f = {xi[j]:.3f})')
for ax in axes[-1]:
    ax.set_xlabel(r'Re $W_j$ / $\sigma_j$')
for ax in axes[:, 0]:
    ax.set_ylabel('density')
fig.tight_layout(); legend(fig, list(sets))
plt.show()

## 3. Spectrum and increment moments
Power spectrum (Hann window, average over signals); skewness $S_3/S_2^{3/2}$ and flatness $S_4/S_2^2$ of the
increments $\delta_\tau u = u(t+\tau) - u(t)$, with $S_p(\tau) = \langle \delta_\tau u^p \rangle$
(non-periodic increments: the data windows are not periodic). Flatness 3 = Gaussian.

In [ ]:
taus = np.unique(np.round(np.logspace(0, np.log10(M // 2), 25)).astype(int))

def spectrum(v):
    w = np.hanning(v.shape[-1])
    return np.fft.rfftfreq(v.shape[-1])[1:], (np.abs(np.fft.rfft((v - v.mean(1, keepdims=True)) * w, axis=1)) ** 2).mean(0)[1:]

def increment_moments(v):
    S = np.array([[np.mean(d ** p) for p in (2, 3, 4)] for d in (v[:, t:] - v[:, :-t] for t in taus)])
    return S[:, 1] / S[:, 0] ** 1.5, S[:, 2] / S[:, 0] ** 2

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for name, v in sets.items():
    f, P = spectrum(v)
    skew, flat = increment_moments(v)
    axes[0].loglog(f, P, **STYLE[name])
    axes[1].semilogx(taus, skew, **STYLE[name])
    axes[2].semilogx(taus, flat, **STYLE[name])
axes[0].set(xlabel='frequency (1/sample)', ylabel='power', title='power spectrum')
axes[1].set(xlabel=r'$\tau$ (samples)', ylabel=r'$S_3 / S_2^{3/2}$', title='skewness of increments')
axes[2].set(xlabel=r'$\tau$ (samples)', ylabel=r'$S_4 / S_2^2$', title='flatness of increments')
axes[1].axhline(0, color='0.5', lw=0.8); axes[2].axhline(3, color='0.5', lw=0.8, ls=':')
fig.tight_layout(); legend(fig, list(sets))
plt.show()

## 4. Signed leverage
Bouchaud, Matacz & Potters (2001):
$L(\tau) = \langle \delta u(t-\tau)\, \delta u(t)^2 \rangle / \langle \delta u^2 \rangle^2$, with
$\delta u(t) = u(t) - u(t-s)$ at scale $s$ ($s = 1$ is the paper's definition). $L = 0$ if $u$ and $-u$ are
equally likely; $L(-\tau) = -L(\tau)$ if the signal is time-reversible. Band = standard error over signals.

In [ ]:
scales, max_tau = (1, 4, 16), 60
fig, axes = plt.subplots(1, len(scales), figsize=(4.4 * len(scales), 3.6), sharey=True)
for ax, s in zip(axes, scales):
    for name, v in sets.items():
        tau, L, err = signed_leverage_correlation(v, scale=s, max_tau=max_tau)
        ax.plot(tau, L, **STYLE[name])
        ax.fill_between(tau, L - err, L + err, color=STYLE[name]['color'], alpha=0.15, lw=0)
    ax.axhline(0, color='0.5', lw=0.8); ax.axvline(0, color='0.5', lw=0.8)
    ax.set(xlabel=r'$\tau$ (samples)', title=f'scale s = {s}')
axes[0].set_ylabel(r'$L(\tau)$')
fig.tight_layout(); legend(fig, list(sets))
plt.show()